# Stat 220, Unit 4 Homework: Prediction and Choosing Predictors

`delivery_routes.csv` is 500 completed delivery routes. `minutes` is how long the
route took. The predictors that make sense are `stops`, `packages`, `miles`, `downtown`, and
`rain`. The company also records `van_age_years`, `dispatcher_rating`, and `month`.

Run the next two cells first. The first one installs what you need, and the second loads the
data that everything else here uses.

In [1]:
%pip install -q numpy pandas statsmodels scikit-learn matplotlib


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split

routes = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/delivery_routes.csv")
folds = KFold(5, shuffle=True, random_state=0)
routes.head()

,minutes,stops,packages,miles,downtown,rain,van_age_years,dispatcher_rating,month
0,254.3,49,105,55.6,1,0,8,3,7
1,153.5,24,71,32.0,0,0,5,5,2
2,450.9,73,226,55.4,1,1,7,4,4
3,387.1,71,175,73.2,1,0,1,1,12
4,111.2,12,18,32.1,0,1,7,1,11


**Problem 1.** *Planning tomorrow's schedule.* Tomorrow's route has 45 stops, 110 packages,
38 miles, goes downtown, and no rain is forecast.

Part a. Report the predicted minutes, the confidence interval, and the prediction interval. Say in one sentence what each of the two intervals is about.

In [3]:
fit = smf.ols("minutes ~ stops + packages + miles + downtown + rain", data=routes).fit()

new = pd.DataFrame({"stops": [45], "packages": [110], "miles": [38],
                    "downtown": [1], "rain": [0]})
print(fit.get_prediction(new).summary_frame(alpha=0.05).round(1))

    mean  mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  obs_ci_upper
0  273.9      2.1          269.7          278.1         229.9         317.8


_Your answer:_

Predicted minutes: 273.9

Mean_CI: [269.7, 278.1]

Predicted_CI = [229.9, 317.8]

The mean interval is the interval the mean of the predicted data is between. Because there is a higher chance of being wrong about a single number, giving a single mean is less confident, while an interval gives more confidence. The observed confidence interval is the interval a single predicted data point could fall between.



Part b. The dispatcher has to tell one driver when to expect to finish tomorrow. Which of the two intervals should they use, and why?

_Your answer:_

Because he's asking about a single point, it's best to give the observed data point: [229.9, 317.8]


Part c. The operations manager asks a different question: across all the downtown routes like this one, what is our average time? Which interval answers that one?

_Your answer:_

Because he is asking about the data with these circumstances as a whole, he should give the Mean CI which is between 269.7 to 278.1


**Problem 2.** *A route that looks ordinary.* A planner asks for a prediction for a route with
25 stops and 120 packages.

Part a. Each value is common on its own. Report how many completed routes are near each value, and how many are near both.

In [4]:
near_stops = routes.stops.between(20, 30)
near_packages = routes.packages.between(110, 130)

print("routes near this stop count   :", near_stops.sum())
print("routes near this package count:", near_packages.sum())
print("routes near both              :", (near_stops & near_packages).sum())
print("correlation between stops and packages:", round(routes.stops.corr(routes.packages), 2))

routes near this stop count   : 86
routes near this package count: 59
routes near both              : 0
correlation between stops and packages: 0.92


_Your answer:_

routes near this stop count   : 86

routes near this package count: 59

routes near both              : 0





Part b. In three or four sentences, explain what your part a numbers mean for a prediction for this route, and what you would tell the planner who asked for it.

_Your answer:_

There seems to be a 0.92 correlation between stops and packages, so as stops increases so does packages. In the example of 25 stops and 120 packages, because 25 stops is so less than 120, there are 0 routes that have both. That is also why it's common to have 25 stops or 120 packages, but not both together.


**Problem 3.** *How wrong will it be?* A new depot has 50 completed routes so far, and the
manager wants one number for how far off a time estimate typically is.

Part a. Report both numbers, and say which one you would give the manager.

In [5]:
depot = routes.sample(50, random_state=3)
X = depot[["stops", "packages", "miles", "downtown", "rain"]]
y = depot["minutes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
m = LinearRegression().fit(X_train, y_train)

print("on the rows it was fitted on:", round(np.sqrt(((y_train - m.predict(X_train))**2).mean()), 1))
print("on the rows held back       :", round(np.sqrt(((y_test - m.predict(X_test))**2).mean()), 1))

on the rows it was fitted on: 22.9
on the rows held back       : 27.7


_Your answer:_

on the rows it was fitted on: 22.9
on the rows held back       : 27.7

I would give the number 27.7 which is the RMSE on the rows held back. 


Part b. Explain in two or three sentences why those two numbers differ, and which way the difference would go if the depot had 500 routes instead of 50.

_Your answer:_

The first number is the training error, how off the model was during training, and the second is testing error, how off the model was when given new data. If the depot had 50 routes instead of 500, the testing error would increase because the model would have less data to be trained on, and be less accurate in predicting new data.

**Problem 4.** *Which predictors belong?* Still the 50-route depot.

Part a. Report the cross-validated error, the $R^2$, and the AIC for each set. Say which set you would ship.

In [6]:
sets = {
    "stops only": ["stops"],
    "the five that make sense": ["stops", "packages", "miles", "downtown", "rain"],
    "those five plus three junk": ["stops", "packages", "miles", "downtown", "rain",
                                   "van_age_years", "dispatcher_rating", "month"],
}
for name, cols in sets.items():
    mse = -cross_val_score(LinearRegression(), depot[cols], depot.minutes, cv=folds,
                           scoring="neg_mean_squared_error").mean()
    f = smf.ols("minutes ~ " + " + ".join(cols), data=depot).fit()
    print(f"{name:<28} CV error {np.sqrt(mse):5.1f}   R2 {f.rsquared:.4f}   AIC {f.aic:.1f}")

stops only                   CV error  38.1   R2 0.9165   AIC 503.7
the five that make sense     CV error  27.3   R2 0.9625   AIC 471.7
those five plus three junk   CV error  31.1   R2 0.9646   AIC 474.8


_Your answer:_

stops only                   CV error  38.1   R2 0.9165   AIC 503.7
the five that make sense     CV error  27.3   R2 0.9625   AIC 471.7
those five plus three junk   CV error  31.1   R2 0.9646   AIC 474.8

I would ship the 'five that make sense' set because it has less CV error and lower AIC score.

Part b. One set has the highest $R^2$ and is still not the set you would ship. Explain what $R^2$ is doing here in two or three sentences.

_Your answer:_

R squared tells how well a model is predicting the outcome. It usually increase or stays the same whenever you add any new predictor to the model, so it can sometimes reward overfitting. When looking at the AIC of the model with the highest R squared, it shows that it's not using the extra predictors efficiently. 

Part c. A colleague wants to add a squared term in `stops` on top of the five. Say what you would check before agreeing, and what you would expect with only 50 routes.

_Your answer:_



**Problem 5.** *What can and cannot be said.* No computer.

Part a. A colleague ran stepwise selection over 65 columns, kept the 11 with p-values under 0.05, and reports an $R^2$ of 0.96 with every predictor significant. Write the two or three sentences you would say in response.

_Your answer:_

Because this colleague hand picked only the significant values, the model is trained in random noise data, and is likely overfitted. If the model were to predict a new unseen data point, it probably would predict it terribly. 

Part b. A lasso fit sends `rain` to exactly zero. Does that show rain has no effect on how long a route takes? Explain.

_Your answer:_

No, it does not mean it doesn't have any effect on routes. It can mean that compared to the other predictors, rain may have less of an effect. Lasso fit seeks to simply the model, so the delay that rain gives on routes might be too small to statistically impact the output.


Part c. The model predicts well and `packages` has a large coefficient. The manager asks whether splitting the same deliveries into more packages would make routes take longer. Answer in two or three sentences.

_Your answer:_

No, because a large coefficient does not represent cause and effect. This simply states the relationship observed between routes and packages, but splitting the same delivery in more packages will keep the same amount of cargo, so it probably won't change the route as much. The amount of packages is not the causal effect of longer routes, but a perceived relationship. 
